# Assignment 7

**Group members** :
- Max Chipani
- Jesus Gamboa
- Karen Salazar
- Paolo Gutierrez
- Luis Camarena

In [ ]:
#pip install geopandas
#pip install matplotlib
#pip install chardet

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
import pandas as pd
import numpy as np
import geopandas as gpd
import chardet

## Question 1

Import the data located at this link. It has information on people infected with dengue at the district level for 2015 to 2021.

In [ ]:
# Getting the character format (encoding type)
base = open(r'../../_data/data_dengue_peru.csv', 'rb').read()
det = chardet.detect(base)
charenc = det['encoding']
charenc

In [ ]:
# Getting data of people infected with dengue
data = pd.read_csv(r'../../_data/data_dengue_peru.csv', dtype={'Ubigeo': 'str'}, encoding=charenc)
data.sort_values(by=['Año', 'Ubigeo', 'Semana'], inplace=True)
data.reset_index(inplace=True)
data.head()

In [ ]:
# Getting data types of dataframe
data.dtypes

## Question 2

Generate ubigeo for Departments and Provinces taking the first two and four numbers. Hint: Use this code.

In [ ]:
# Generating the ubigeo variable
data['codgeo'] = data['Ubigeo'].astype(str).str.zfill(6)
data['codgeo'].head()

In [ ]:
# Generating the code of departaments
data['cod_dep'] = data['codgeo'].str[:2]
data['cod_dep'].value_counts()

In [ ]:
# Generating the cod of provinces
data['cod_pro'] = data['codgeo'].str[:4]
data['cod_pro'].head()

## Question 3

Use geopandas to plot the number of cases in 2021 by the district using a continuous legend. Do not forget to indicate the color of NA values. Use this shapefile.

In [ ]:
data['Casos'] = pd.to_numeric(data['Casos'], errors='coerce').astype('Int64') 
data.head(3)

In [ ]:
# Filter the DataFrame 'data' to include only the rows where the 'Año' column has the value 2021.
data_2021 = data.loc[data.Año==2021]

# Display the first 3 rows of the DataFrame 'data_2021'
data_2021.head(3)

In [ ]:
# Group the DataFrame 'data_2021' by the 'Ubigeo' column and compute the sum of the 'Casos' column for each group.
# After aggregation, reset the index to turn the grouped columns back into regular columns, resulting in 'data_2021_anual'.
data_2021_anual = data_2021.groupby('Ubigeo')['Casos'].sum().reset_index()

# Display the DataFrame
data_2021_anual

In [ ]:
# Read the shapefile from the specified path into a GeoDataFrame 'shpf1'.
shpf1 = gpd.read_file( r'../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp' )

# Select only the 'UBIGEO' and 'geometry' columns from the GeoDataFrame 'shpf1'.
shpf1 = shpf1[['UBIGEO', 'geometry']]

# Display the filtered GeoDataFrame 'shpf1'.
shpf1

In [ ]:
# Access the coordinate reference system (CRS) of the GeoDataFrame shpf1
shpf1.crs

In [ ]:
# Perform a left merge between the GeoDataFrame 'shpf1' and the DataFrame 'data_2021_anual',
# merging them based on the 'UBIGEO' column from 'shpf1' and the 'Ubigeo' column from 'data_2021_anual'.
# The 'how="left"' ensures that all rows from 'shpf1' will be included, even if there's no match in 'data_2021_anual'.
data_map1 = pd.merge(shpf1, data_2021_anual, how="left", left_on="UBIGEO", right_on="Ubigeo")
data_map1

In [ ]:
# Create a figure and axes object for the plot, setting the size to 20x20 inches
fig, ax = plt.subplots(figsize=(20, 20))

# Plot the 'Casos' column from the 'datos_map1' GeoDataFrame on the axes, 
# using a 'Reds' colormap, gray borders, and a line width of 0.5. 
# The legend is also enabled.
data_map1.plot(column    ='Casos', 
                cmap      ='Reds', 
                edgecolor ='gray', 
                linewidth =0.5,
                ax        =ax,
                legend    =True)

# Create a mask to identify rows where the 'Casos' column has missing values (NaN)
nan_mask = data_map1['Casos'].isna()

# Plot the areas with missing 'Casos' values (NaN) on the same axes,
# coloring them in light grey with gray borders and no legend
data_map1[nan_mask].plot(ax=ax, color='lightgrey', edgecolor='gray', linewidth=0.5, legend=False)

# Display the plot
plt.show()

## Question 4

Use geopandas to plot the number of cases in 2021 by the province using a continuous legend. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the province level.

In [ ]:
# Group 'data_2021' by 'cod_pro', sum the 'Casos' for each group, and reset the index.
# Display the resulting DataFrame
data_2021_provincial = data_2021.groupby('cod_pro')['Casos'].sum().reset_index()
data_2021_provincial

In [ ]:
# Extracts the first 4 characters of 'UBIGEO' and creates the 'UBIGEO_Prov' column with the province code.
shpf1['UBIGEO_Prov'] = shpf1['UBIGEO'].str[:4]
# Perform a spatial dissolve operation based on the new provincial code. This combines polygons that belong to the same province
shpf1_provincial = shpf1.dissolve(by='UBIGEO_Prov')

In [ ]:
# Show the first 3 rows of the provincial GeoDataFrame
shpf1_provincial.head(3)

In [ ]:
# Show the first 3 rows of the 2021 provincial data DataFrame
data_2021_provincial.head(3)

In [ ]:
data_map_prov = pd.merge(shpf1_provincial, data_2021_provincial, how="left", left_on="UBIGEO_Prov", right_on="cod_pro")
# Define the color for NaN values
color_nan = 'lightgrey'

# Create the plot
fig, ax = plt.subplots(figsize=(20, 20))

# Plot with the desired colormap
data_map_prov.plot(column='Casos', cmap='Reds', 
                    edgecolor='gray', 
                    linewidth=0.5,
                    ax=ax,
                    legend=True)

# Plot the NaN values with the specified color
nan_mask = data_map_prov['Casos'].isna()
data_map_prov[nan_mask].plot(ax=ax, color=color_nan, edgecolor='gray', linewidth=0.5, legend=False)

## Question 5

Use geopandas to plot the number of cases by the department for all the years using subplots. Every subplot for each year. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the department level.

In [ ]:
# Aggregate the data by department and year
data_dep = data.groupby(['cod_dep', 'Año'], as_index=False)['Casos'].sum()
data_dep

In [ ]:
# Upload shape file at district level
map = gpd.read_file(r'../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp')

In [ ]:
# Counting the number of unique years present in the 'Año' column of the data_dep DataFrame.
data_dep['Año'].value_counts().size

In [ ]:
# Aggregating the geometries and data by 'IDPROV' (likely the province ID). This combines
# all geometries that belong to the same province into a single geometry.
map_dep = map.dissolve(by='IDPROV')
# Further aggregating the geometries and data by 'CCDD' (likely the department ID). This
# step merges all geometries within the same department into one.
map_dep = map_dep.dissolve(by='CCDD')
map_dep

In [ ]:
map_dep = pd.merge(map_dep, data_dep, left_on='CCDD', right_on='cod_dep', how='left')

In [ ]:
# Assuming map_dep contains district-level data
fig, axis = plt.subplots(nrows=3, ncols=3, figsize=(15, 15))

x = 0
for i in range(3):
    for j in range(3):
        if x < 7:
            ax = axis[i][j]
            año = map_dep['Año'].unique()[x]
            # Filter by year and plot by district (make sure your GeoDataFrame contains district-level data)
            map_dep_año = map_dep[map_dep['Año'] == año]
    
            # Check if the GeoDataFrame for the year is empty
            if map_dep_año.empty:
                ax.set_title(f"Year {int(año)}: No data", fontsize=10)
                ax.axis("off")  # Turn off axis if no data
            else:
                # Plot the map at district level
                map_dep_año.plot(column='Casos', 
                                 cmap='Blues', 
                                 linestyle='-', 
                                 edgecolor='black', 
                                 legend=(x == 0),  # Show legend only for the first plot
                                 missing_kwds=dict(color="#DADADB"),
                                 ax=ax)
                ax.set_title(f"Year {int(año)}", fontsize=12)
            
            x += 1
        else:
            fig.delaxes(axis[i, j])  # Remove unused subplots if no more years to plot

# Adjust layout
plt.tight_layout()
plt.show()




## Question 6

Use geopandas to plot the number of cases by the department for all 2021 quarters using subplots. Every subplot for each quarter. Use a categorical legend with 5 bins. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the department level. Hint: Use Semana variable to group by quarters.

In [ ]:
# Create a 'Trimestre' column to represent the quarters
# Calculate the trimester by dividing the week number by 13 and adjusting the result
data_2021 = data_2021.copy()
data_2021.loc[:, 'Trimestre'] = (data_2021['Semana'] - 1) // 13 + 1
data_2021.loc[:,'Trimestre']  = data_2021['Trimestre'].replace(5, 4)
data_2021.head(3)

In [ ]:
# Aggregate the data by trimester and department
data_trimestral = data_2021.groupby(['Trimestre', 'cod_dep'])['Casos'].sum().reset_index()

# Get unique trimesters for the plotting
unique_trimesters = data_trimestral['Trimestre'].unique()
unique_trimesters

# Extract the first 2 characters from the 'UBIGEO' column to create the 'UBIGEO_Dept' column
# The first 2 digits of the 'UBIGEO' code represent the department
shpf1['UBIGEO_Dept'] = shpf1['UBIGEO'].str[:2]

# Perform a spatial dissolve operation based on the new 'UBIGEO_Dept' column
# This will merge polygons that belong to the same department, creating one for each department
shpf1_departamental = shpf1.dissolve(by='UBIGEO_Dept')

# Display the first few records of the new GeoDataFrame to verify the operation
print(shpf1_departamental.head())



In [ ]:

#  Create subplots in a grid of 1 row and 4 columns (one for each trimester)

fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(18, 10))  

# Define the categorical color scale

bins = [0, 500, 1000, 1500, 2000, 2500] 
cmap = plt.get_cmap('Blues', len(bins)-1) 
norm = BoundaryNorm(boundaries=bins, ncolors=len(bins)-1) 

# Loop over each unique trimester to create a subplot
for i, trimestre in enumerate(unique_trimesters):
    ax = axes.flatten()[i]  # Asegura que se use el índice correcto en la cuadrícula
    data_trimestre = data_trimestral[data_trimestral['Trimestre'] == trimestre]
    
    # Merge trimester data with the shapefile
    data_map_dept = pd.merge(shpf1_departamental, data_trimestre, how="left", left_on="UBIGEO_Dept", right_on="cod_dep")
    
    # Plot the data on the map
    data_map_dept.plot(ax=ax,
                        column='Casos',
                        cmap=cmap,
                        norm=norm,
                        edgecolor='gray',
                        linewidth=0.5,
                        linestyle='--',
                        legend=True,
                        classification_kwds=dict(bins=bins),
                        legend_kwds={
                            'label': "Número de Casos",
                            'orientation': "horizontal"
                        })
    
    # Indicate the color for NaN value
    nan_mask = data_map_dept['Casos'].isna()
    data_map_dept[nan_mask].plot(ax=ax, color='lightgrey', edgecolor='gray', linewidth=0.5, legend=False)
    
    # Set the title for the subplot
    ax.set_title(f'Trimestre {trimestre} - 2021', fontsize=15)
    ax.axis('off')  

# Adjust the spacing between subplots
plt.tight_layout()
plt.show()